# Stage 08 — Read-only Analytics Serving

**Goal:** Calculate correct analytics metrics and enforce a read-only data-serving boundary.

**Setup:** Python 3.11+, Jupyter/VS Code; local synthetic examples only.

**CS bridge:** Explain the invariant and its failure mode before running code.

**Source:** https://www.sqlite.org/pragma.html#pragma_query_only

**Next:** Complete the graded lab under ../labs/ and pass an unseen transfer challenge.

## 1. Predict before running

Before selecting Run All, write down what the next assertion should prove and what it does **not** verify. Run the cell only after making your prediction.

In [ ]:
import sqlite3
db=sqlite3.connect(":memory:")
db.execute("create table events(id text primary key,magnitude real)")
db.executemany("insert into events values (?,?)",[("A",2.0),("B",None),("C",4.0)])
db.commit()
total,measured,average=db.execute("select count(*),count(magnitude),avg(magnitude) from events").fetchone()
assert total==3 and measured==2 and average==3.0
print("Events:",total,"known magnitudes:",measured,"average:",average)

## 2. Independent failure check

On your own practice branch, change one parameter. Explain why the assertion fails or the output changes, then restore the original value. Preserve the invariant.

In [ ]:
db.execute("pragma query_only=on")
assert db.execute("select count(*) from events").fetchone()[0]==3
try:
    db.execute("update events set magnitude=10")
    raise AssertionError("write operation must fail")
except sqlite3.OperationalError:
    print("Read-only boundary succeeded; an HTTP service also needs allowlisting and auth design.")
db.close()

## 3. Unseen transfer (no AI)

Design an endpoint returning regional summaries with validated parameters and a bounded LIMIT. What internal data must never be exposed?

**Do not mark the stage complete based on this notebook alone.** Finish the corresponding graded lab and complete a mentor-reviewed transfer task.